# Phase 5 — ML Engine Validation Notebook

This notebook validates the **production** `backend/ml_engine` package using local synthetic data only.

**No network access, no API keys, and no production logic in this notebook** — every computation is
a call into `backend/ml_engine/*` or `backend/data_engine/*`.

Contents:

1. Setup / path bootstrap
2. The 27-column feature contract and the leakage guard
3. Trailing-only rule baseline
4. Isolation Forest, thresholding and anomaly scores
5. Baseline vs. model evaluation (**honest result**)
6. SHAP driver attribution and the additivity assertion
7. Determinism of attribution (including an RNG burn)
8. Known score semantics: batch-dependent rank-CDF
9. Missing-data behavior

**Anomaly is not financial risk.** An ML anomaly marks unusual multi-ratio behaviour; it is not a
Phase 4 risk score, it is never mapped onto the Phase 4 0–100 scale, and an attribution explains
the model, not the business cause.

In [1]:
import sys
from pathlib import Path

# Path bootstrap so a fresh kernel can import the repository package.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from backend.data_engine.contracts import (
    AnomalyInjection,
    AnomalyType,
    SyntheticCompanyConfig,
)
from backend.data_engine.ingest.synthetic import _anomaly_active, generate_company
from backend.ml_engine.baseline import rolling_rule_baseline
from backend.ml_engine.contracts import FEATURE_SCHEMA_VERSION
from backend.ml_engine.evaluate import evaluate_methods
from backend.ml_engine.explain import attribute_row, explain_flagged_period
from backend.ml_engine.features import (
    FEATURE_NAMES,
    FORBIDDEN_COLUMNS,
    build_feature_frame,
    fingerprint_frame,
)
from backend.ml_engine.models import fit_isolation_forest, threshold_from_expected_rate

print("Repository root:", REPO_ROOT)
print("Feature schema version:", FEATURE_SCHEMA_VERSION)
print("Expected feature count:", len(FEATURE_NAMES))

Repository root: E:\AI Business Risk
Feature schema version: 1.0.0
Expected feature count: 27


## 1. Synthetic fixture (local, deterministic)

A single synthetic company with a **labelled `MARGIN_COLLAPSE` injection** over months 18–21.
This is the same fixture shape as the committed ML golden (`seed=7101`), so the notebook output is
directly comparable to `backend/tests/golden/ml_anomaly_golden.json`.

Injected labels are **evaluation-only**: they are never features.

In [2]:
SEED = 7101
EXPECTED_RATE = 4 / 24  # match the golden fixture threshold policy

config = SyntheticCompanyConfig(
    seed=SEED,
    periods=24,
    inject_anomalies=[
        AnomalyInjection(
            type=AnomalyType.MARGIN_COLLAPSE,
            start_month=18,
            duration_months=4,
            magnitude=0.6,
        )
    ],
)
company = generate_company(config)
labels = [_anomaly_active(config.inject_anomalies[0], i + 1) for i in range(24)]

print(f"Company: {company.profile.name} | Sector: {company.profile.sector}")
print(f"Periods: {len(company.periods)}")
print("Injected anomaly rows (0-based):", [i for i, y in enumerate(labels) if y])

Company: Fairmount Industries | Sector: manufacturing
Periods: 24
Injected anomaly rows (0-based): [17, 18, 19, 20]


## 2. Feature contract (27 columns) and the leakage guard

The feature matrix is built from **Phase 4 metric calculators** — the ML engine reuses the frozen
risk formulas and adds no new financial mathematics.

Composition:

- **14 base ratios** (gross/operating/net margin, ROA, ROE, debt-to-equity, debt-to-EBITDA,
  current/quick/cash ratio, short-term coverage, DSO, DIO, DPO)
- **+ 1** cash conversion cycle (`ccc`)
- **+ 6 `_delta`** period-over-period changes of six key ratios
- **+ 6 `_rollvol`** rolling volatility (window 12, minimum 8 observations)

Every feature is **scale-free**. Identifiers, timestamps, seeds, labels and metadata can never
enter the matrix — enforced by `FORBIDDEN_COLUMNS`.

In [3]:
frame = build_feature_frame(company.periods)

print("Frame shape:", frame.shape)
print("Columns match the frozen contract:", list(frame.columns) == list(FEATURE_NAMES))
print()
print("Columns:")
for name in FEATURE_NAMES:
    print("  -", name)

# Leakage guard: no identifier / metadata / label column may appear.
leaked = set(frame.columns) & set(FORBIDDEN_COLUMNS)
print()
print("Leaked columns:", sorted(leaked) or "none")
assert not leaked, f"leakage: {sorted(leaked)}"
print("PASS - leakage guard holds")

Frame shape: (24, 27)
Columns match the frozen contract: True

Columns:
  - gross_margin
  - operating_margin
  - net_margin
  - roa
  - roe
  - debt_to_equity
  - debt_to_ebitda
  - current_ratio
  - quick_ratio
  - cash_ratio
  - st_obligation_coverage
  - dso
  - dio
  - dpo
  - ccc
  - gross_margin_delta
  - operating_margin_delta
  - net_margin_delta
  - roa_delta
  - current_ratio_delta
  - dso_delta
  - gross_margin_rollvol
  - operating_margin_rollvol
  - net_margin_rollvol
  - roa_rollvol
  - current_ratio_rollvol
  - dso_rollvol

Leaked columns: none
PASS - leakage guard holds


In [4]:
# Early periods have no trailing history, so delta/rollvol features are missing there
# and are therefore excluded from model training (see section 4).
head = frame.head(3)[[c for c in frame.columns if c.endswith(("_delta", "_rollvol"))]]
print("First three periods, history-derived features:")
print(head.to_string())

complete_rows = int(frame.dropna().shape[0])
print()
print(f"Complete rows available for training: {complete_rows} / {len(frame)}")

# Stable fingerprint of the exact matrix the model consumes.
print("Feature fingerprint:", fingerprint_frame(frame, config={"seed": SEED})[:32], "...")

First three periods, history-derived features:
   gross_margin_delta  operating_margin_delta  net_margin_delta  roa_delta  current_ratio_delta     dso_delta  gross_margin_rollvol  operating_margin_rollvol  net_margin_rollvol  roa_rollvol  current_ratio_rollvol  dso_rollvol
0                 NaN                     NaN               NaN        NaN                  NaN           NaN                   NaN                       NaN                 NaN          NaN                    NaN          NaN
1       -7.105427e-15                0.545983          0.791035   0.116739             0.042051 -1.136868e-13                   NaN                       NaN                 NaN          NaN                    NaN          NaN
2        7.105427e-15               -0.193999         -0.012438  -0.019510             0.091800  1.136868e-13                   NaN                       NaN                 NaN          NaN                    NaN          NaN

Complete rows available for training: 13 / 2

## 3. Rule baseline (trailing window only)

The comparison baseline is a deterministic rolling z-score / Tukey-IQR rule over the **same**
feature matrix. It only ever looks *backwards* (`frame.iloc[max(0, i-window) : i]`), so it cannot
benefit from future periods.

A period is flagged when `|z| >= 3.0` **or** the value falls outside
`Q1 - 1.5*IQR .. Q3 + 1.5*IQR`, given at least 8 trailing observations. Early periods therefore
cannot be flagged.

In [5]:
rule = rolling_rule_baseline(frame)

print("Rule baseline flags (True = flagged):")
for i, flag in enumerate(rule.flags):
    if flag:
        print(f"  row {i:>2} | score {rule.scores[i]:>8.4f} | breached {rule.breached_features[i]}")

print()
print("Row 0 flag (no trailing history):", rule.flags[0])
assert rule.flags[0] is False, "trailing-only rule must not flag the first period"
print("PASS - baseline is trailing-only")

Rule baseline flags (True = flagged):
  row  8 | score   2.8781 | breached ['debt_to_equity', 'net_margin', 'operating_margin', 'roa', 'roe']
  row  9 | score   1.8237 | breached ['operating_margin_delta']
  row 10 | score   2.2212 | breached ['current_ratio_delta', 'debt_to_equity', 'operating_margin']
  row 13 | score   2.5578 | breached ['current_ratio_delta']
  row 14 | score   1.5218 | breached ['roa_delta']
  row 17 | score 10178211053721096.0000 | breached ['current_ratio', 'current_ratio_delta', 'current_ratio_rollvol', 'debt_to_equity', 'gross_margin', 'gross_margin_delta', 'gross_margin_rollvol', 'net_margin', 'net_margin_delta', 'net_margin_rollvol', 'operating_margin', 'operating_margin_delta', 'operating_margin_rollvol', 'roa', 'roa_delta', 'roa_rollvol', 'roe', 'st_obligation_coverage']
  row 18 | score   4.3435 | breached ['current_ratio_rollvol', 'debt_to_equity', 'dso_delta', 'gross_margin', 'gross_margin_rollvol', 'net_margin', 'net_margin_rollvol', 'operating_margin'

## 4. Isolation Forest, thresholding and anomaly scores

- Fitted with `n_estimators=200`, `random_state=42`, `n_jobs=1` on **complete rows only**.
- Features are median/IQR-scaled (robust to outliers, IQR floored at 1.0).
- The **operating threshold** is a quantile of the training decision distribution at the expected
  flag rate — not a hand-picked constant.
- `anomaly_scores()` returns a **rank-CDF in [0, 1]**, higher = more anomalous. It is *not* a
  probability and is *not* the Phase 4 0–100 risk score.

In [6]:
fitted = fit_isolation_forest(frame)
threshold = threshold_from_expected_rate(fitted.train_decision, EXPECTED_RATE)
scores = fitted.anomaly_scores(frame)
decisions = fitted.decision_values(frame)
model_flags = [(-v) >= threshold for v in decisions]

print(f"Fitted on {len(fitted.feature_names)} features, threshold = {threshold:.12f}")
print()
print("  row |  decision_function | anomaly_score | model_flag | injected_label")
for i in range(len(frame)):
    print(
        f"  {i:>3} | {decisions[i]:>17.9f} | {scores[i]:>13.6f} "
        f"| {str(model_flags[i]):>10} | {str(labels[i]):>14}"
    )

assert all(0.0 <= s <= 1.0 for s in scores)
print()
print("PASS - all anomaly scores are within [0, 1]")

Fitted on 27 features, threshold = 0.017591369042

  row |  decision_function | anomaly_score | model_flag | injected_label
    0 |       0.047287858 |      0.270270 |      False |          False
    1 |      -0.001949658 |      0.648649 |      False |          False
    2 |       0.023768422 |      0.432432 |      False |          False
    3 |       0.036731067 |      0.351351 |      False |          False
    4 |       0.055736761 |      0.135135 |      False |          False
    5 |       0.012006545 |      0.513514 |      False |          False
    6 |      -0.015970841 |      0.729730 |      False |          False
    7 |       0.002540001 |      0.621622 |      False |          False
    8 |      -0.003212181 |      0.702703 |      False |          False
    9 |       0.039630451 |      0.324324 |      False |          False
   10 |       0.049154617 |      0.243243 |      False |          False
   11 |       0.029881048 |      0.405405 |      False |          False
   12 |     

## 5. Baseline vs. model — the honest result

Both methods are scored against the **same injected labels** with the same frozen metric set
(ROC-AUC, PR-AUC, precision/recall/F1/FPR/FNR at the threshold, precision@k and recall@k).

Decision **D4** states the Isolation Forest is retained *only if it beats the baseline*. The verdict
below is therefore the actual falsifiability outcome, not a marketing number.

In [7]:
report = evaluate_methods(
    model_scores=scores,
    model_flags=model_flags,
    baseline_scores=rule.scores,
    baseline_flags=rule.flags,
    labels=labels,
    dataset_fingerprint=fingerprint_frame(frame, config={"seed": SEED}),
    threshold_policy=f"train quantile at expected rate {EXPECTED_RATE}",
)

m, b = report.model, report.baseline
print(f"Verdict: {report.verdict}")
print(f"Detail: {report.detail}")
print()
print(f"{'metric':<24}{'Isolation Forest':>20}{'rule baseline':>20}")
print("-" * 64)
print(f"{'ROC-AUC':<24}{m.roc_auc:>20.4f}{b.roc_auc:>20.4f}")
print(f"{'PR-AUC':<24}{m.pr_auc:>20.6f}{b.pr_auc:>20.6f}")
for key in ("precision", "recall", "f1", "fpr", "fnr"):
    print(f"{key:<24}{m.metrics_at_threshold[key]:>20.4f}{b.metrics_at_threshold[key]:>20.4f}")
for key in sorted(m.at_k):
    print(f"{key:<24}{m.at_k[key]:>20.4f}{b.at_k[key]:>20.4f}")

print()
print("Limitation:", report.limitation)
print()
if report.verdict == "baseline_wins_or_tie":
    print("HONEST RESULT: on this fixture the rolling rule baseline BEATS the Isolation Forest.")
    print("Per D4 the model is therefore NOT retained. No accuracy claim is made.")
else:
    print("On this fixture the Isolation Forest beats the baseline on both PR-AUC and F1.")

Verdict: baseline_wins_or_tie
Detail: {'n_periods': 24, 'n_injected': 4}

metric                      Isolation Forest       rule baseline
----------------------------------------------------------------
ROC-AUC                               0.9000              0.9500
PR-AUC                              0.525000            0.830357
precision                             0.5714              0.3636
recall                                1.0000              1.0000
f1                                    0.7273              0.5333
fpr                                   0.1500              0.3500
fnr                                   0.0000              0.0000
precision_at_10                       0.4000              0.4000
precision_at_5                        0.6000              0.6000
recall_at_10                          1.0000              1.0000
recall_at_5                           0.7500              0.7500

Limitation: Evaluation uses synthetic injected anomalies only; no real-world pre

### Scope of these numbers

These metrics are **in-sample** on a single 24-period synthetic fixture: the model is fitted on the
same rows it is then scored on. They are **descriptive statistics of the synthetic fixture, not
generalization performance**, and they are not a real-world validity claim.

No external benchmark exists for this system and none is claimed.

## 6. SHAP driver attribution

Attribution uses `shap.Explainer` over the fitted `decision_function`.

**What SHAP 0.52.0 actually selects:** with an `Independent` masker over a generic callable, SHAP
resolves to **`PermutationExplainer`** — a Monte-Carlo permutation estimator. `TreeExplainer` is
*not* called by the production code, because TreeSHAP explains the ensemble's raw path-length
output and therefore does **not** satisfy the engine's additivity contract against
`decision_function`.

A deterministic permutation-importance fallback (`_permutation_row`, `default_rng(42)`) is used if
SHAP is unavailable, raises, or violates additivity.

**Honest naming note:** the internal `explainer` label is `"shap_exact"`, which is technically
misleading — the permutation estimator is *approximate* (values differ by ~2e-4 between seeds).
The label is left unchanged in Phase 5 because renaming it would invalidate the committed golden.

In [8]:
# Explain the highest-scoring complete row.
complete = frame.dropna().reset_index(drop=True)
fitted_complete = fit_isolation_forest(complete)
complete_scores = fitted_complete.anomaly_scores(complete)
row_index = int(np.argmax(complete_scores))

explanation = explain_flagged_period(
    fitted_complete,
    complete,
    row_index,
    company_id=str(company.edgar_cik),
    top_k=5,
)

print(f"Explaining complete-row {row_index} | anomaly_score = {complete_scores[row_index]:.6f}")
print(f"explainer          : {explanation.explainer}")
print(f"explainer_version  : {explanation.explainer_version}")
print(f"background         : {explanation.background}")
print()
print(f"{'rank':>4}  {'feature':<28}{'feature_value':>16}{'attribution':>14}  direction")
print("-" * 86)
for rank, driver in enumerate(explanation.drivers, start=1):
    value = "n/a" if driver.feature_value is None else f"{driver.feature_value:.6f}"
    print(
        f"{rank:>4}  {driver.feature:<28}{value:>16}{driver.attribution:>14.6f}  {driver.direction}"
    )

print()
print("Causality note :", explanation.causality_note)
print("Anomaly note   :", explanation.anomaly_note)

E:\AI Business Risk\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PermutationExplainer explainer: 100%|██████████| 1/1 [00:00<?, ?it/s]

PermutationExplainer explainer: 2it [00:10, 10.19s/it]               

Explaining complete-row 10 | anomaly_score = 1.000000
explainer          : shap_exact
explainer_version  : 0.52.0
background         : training matrix (13 rows, max 50 background samples)

rank  feature                        feature_value   attribution  direction
--------------------------------------------------------------------------------------
   1  operating_margin_delta             17.934223     -0.019817  pushes_anomalous
   2  gross_margin_delta                 19.289835     -0.019168  pushes_anomalous
   3  roa_delta                           1.373401     -0.016753  pushes_anomalous
   4  net_margin_delta                   13.382199     -0.014215  pushes_anomalous
   5  net_margin_rollvol                  6.765835     -0.013538  pushes_anomalous

Causality note : Model attribution, not causal explanation: drivers explain the model's output, not the business cause.
Anomaly note   : An ML anomaly flag marks unusual multi-ratio behavior; it is not a financial-risk judgment and 

In [9]:
# Additivity assertion: base_value + sum(values) == decision_function(x) within 1e-6.
attribution = attribute_row(fitted_complete, complete, row_index)
vector = fitted_complete.scaled_matrix(complete[row_index : row_index + 1])
decision = float(fitted_complete.model.decision_function(vector)[0])
reconstructed = attribution.base_value + sum(attribution.values)
error = abs(reconstructed - decision)

print(f"explainer kind      : {attribution.explainer}")
print(f"base_value          : {attribution.base_value:.9f}")
print(f"sum(attributions)   : {sum(attribution.values):.9f}")
print(f"decision_function(x): {decision:.9f}")
print(f"additivity error    : {error:.3e}  (tolerance 1e-6)")
assert error < 1e-6, f"additivity violated: {error}"
print("PASS - SHAP additivity holds")

explainer kind      : shap_exact
base_value          : 0.006828685
sum(attributions)   : -0.131154630
decision_function(x): -0.124325945
additivity error    : 2.776e-17  (tolerance 1e-6)
PASS - SHAP additivity holds


## 7. Determinism of attribution

`shap` 0.52's `PermutationExplainer` samples permutations from the **global NumPy RNG**. Without
explicit control, two attributions of the same row differ (observed drift ~4e-4) and the driver
ranking can reorder.

The production fix has two parts:

1. `shap.Explainer(..., seed=EXPLAINER_SEED)` with `EXPLAINER_SEED = 42`
2. the caller's global RNG state is **saved and restored** around the explainer, so the engine
   never leaks RNG state to its caller

The cell below burns 9,999 random numbers and re-attributes the same row. The same check runs as a
committed regression test in `backend/tests/unit/test_ml_explain.py`.

In [10]:
before = attribute_row(fitted_complete, complete, row_index)
identical_before_burn = attribute_row(fitted_complete, complete, row_index).values == before.values
print("Back-to-back identical (no burn):", identical_before_burn)

# Burn 9,999 numbers from the GLOBAL NumPy RNG.
np.random.seed(12345)
burned = np.random.rand(9999)
print(f"Burned {len(burned)} global NumPy random numbers")

after = attribute_row(fitted_complete, complete, row_index)
print("Identical after the RNG burn:", before.values == after.values)
print("Base value preserved        :", before.base_value == after.base_value)
assert before.values == after.values, "attribution changed after an unrelated RNG burn"
print()
print("PASS - attribution is independent of unrelated RNG consumption")

Back-to-back identical (no burn): True
Burned 9999 global NumPy random numbers


Identical after the RNG burn: True
Base value preserved        : True

PASS - attribution is independent of unrelated RNG consumption


In [11]:
# Caller RNG state must be left untouched by the engine.
np.random.seed(4242)
expected_state = np.random.get_state()[1][:8].tolist()
_ = attribute_row(fitted_complete, complete, row_index)
observed_state = np.random.get_state()[1][:8].tolist()
print("Global RNG state preserved:", expected_state == observed_state)
assert expected_state == observed_state, "engine leaked global RNG state"
print("PASS - global RNG state is restored")

Global RNG state preserved: True
PASS - global RNG state is restored


## 8. Known score semantics: the rank-CDF is batch-dependent

`anomaly_scores()` builds its rank-CDF over the **training rows pooled with the rows being
scored**. The score is therefore a *dataset-relative rank*, not a fixed calibration: scoring a
subset of periods yields different values for the same periods.

This is documented, deliberate behaviour for the current phase and is **not** changed here.
Freezing a reference distribution inside the model artifact is recorded as a deferred decision.

In [12]:
full_rows = fitted.anomaly_scores(frame)[17:22]
subset_rows = fitted.anomaly_scores(frame.iloc[17:22])

print("Rows 17-21 scored within the full 24-row frame:")
print("  ", [round(v, 6) for v in full_rows])
print("Rows 17-21 scored as a 5-row subset:")
print("  ", [round(v, 6) for v in subset_rows])
print()
print("Identical:", [round(v, 6) for v in full_rows] == [round(v, 6) for v in subset_rows])
print("-> documented batch dependence; a frozen reference distribution is deferred")

Rows 17-21 scored within the full 24-row frame:
   [0.891892, 0.810811, 0.837838, 0.864865, 1.0]
Rows 17-21 scored as a 5-row subset:
   [0.833333, 0.666667, 0.722222, 0.777778, 1.0]

Identical: False
-> documented batch dependence; a frozen reference distribution is deferred


## 9. Missing-data behaviour

The engine never fabricates, zero-fills, forward-fills or interpolates financial values.

- a ratio calculator that raises or returns a non-valid status becomes `None` (**not** `0.0`)
- non-finite / non-numeric cells are skipped by the rule baseline
- rows with any missing feature are excluded from model training
- an empty training frame raises `ValueError` instead of silently degrading

In [13]:
from backend.ml_engine.features import _period_dict

dump = _period_dict(company.periods[0])

# (a) every field missing -> every feature missing (None -> NaN), never 0.0
all_missing = {key: None for key in dump}
missing_frame = build_feature_frame([all_missing])
print("All-missing period  -> gross_margin:", missing_frame["gross_margin"].iloc[0])
print("All-missing period  -> ccc          :", missing_frame["ccc"].iloc[0])
assert missing_frame["gross_margin"].isna().all()
print("PASS - missing values stay missing (no fabrication, no zero-fill)")

# (b) a malformed field yields a missing feature, not a crash and not a fake number
malformed = dict(dump)
malformed["revenue"] = "oops"
malformed_frame = build_feature_frame([malformed])
print()
print("Malformed revenue   -> gross_margin:", malformed_frame["gross_margin"].iloc[0])
assert malformed_frame["gross_margin"].isna().all()
print("PASS - malformed input degrades to a missing feature")

# (c) an empty training frame is rejected explicitly
try:
    fit_isolation_forest(pd.DataFrame({"a": []}))
except ValueError as exc:
    print()
    print("Empty training frame -> ValueError:", exc)
    print("PASS - empty training data is rejected, not silently degraded")

All-missing period  -> gross_margin: None
All-missing period  -> ccc          : None
PASS - missing values stay missing (no fabrication, no zero-fill)

Malformed revenue   -> gross_margin: None
PASS - malformed input degrades to a missing feature

Empty training frame -> ValueError: no complete feature rows available for training
PASS - empty training data is rejected, not silently degraded


## Summary of what this notebook demonstrates

| Area | Result |
|---|---|
| Feature contract | 27 scale-free columns, matches `FEATURE_NAMES`, leakage guard holds |
| Rule baseline | trailing-window only; cannot flag the first period |
| Isolation Forest | deterministic, complete rows only, rank-CDF scores in `[0, 1]` |
| Thresholding | training quantile at the expected flag rate |
| Baseline vs. model | `baseline_wins_or_tie` — the rule baseline beats the model on this fixture |
| SHAP attribution | `PermutationExplainer`, additivity < 1e-6, top-k drivers with direction |
| Determinism | attribution identical after a 9,999-number RNG burn; RNG state restored |
| Missing data | missing stays missing; empty training data raises `ValueError` |

**Not claimed:** causal explanation, financial-distress prediction, real-world predictive validity,
or any mapping onto the Phase 4 0–100 risk score.